### Langkah 1: Persiapan Notebook & Cek Perangkat (GPU)

In [1]:
# Install library utama untuk menjalankan model Hugging Face
!pip install -q transformers accelerate sentencepiece torch

import torch
import transformers

# Cek perangkat yang aktif (GPU / CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Perangkat aktif   : {device.upper()}")
if device == "cuda":
    print(f"Nama GPU          : {torch.cuda.get_device_name(0)}")
print(f"Versi PyTorch     : {torch.__version__}")
print(f"Versi Transformers: {transformers.__version__}")

Perangkat aktif   : CPU
Versi PyTorch     : 2.14.1+cpu
Versi Transformers: 5.22.1


### Langkah 2: Menyiapkan Teks Materi Pengembangan (500–800 Kata)

In [2]:
# Masukkan teks bacaan pengembangan (500 - 800 kata)
sumber_bacaan = "Contoh Materi Pengembangan - Transisi Energi dan Tantangan Jaringan Listrik Nasional"

teks_materi = """
Transisi energi menuju sumber terbarukan menjadi salah satu agenda utama pembangunan nasional Indonesia dalam dua dekade terakhir. Pemerintah menargetkan porsi energi baru terbarukan (EBT) dalam bauran energi primer mencapai 23 persen pada tahun 2025 dan meningkat hingga lebih dari 31 persen pada tahun 2050. Target ini ditetapkan untuk menekan emisi gas rumah kaca sekaligus mengurangi ketergantungan pada pembangkit listrik tenaga uap (PLTU) berbasis batu bara yang selama ini mendominasi pasokan listrik nasional hingga lebih dari 60 persen. Secara geografis, Indonesia memiliki potensi energi surya yang sangat melimpah, diperkirakan mencapai lebih dari 200 gigawatt, karena letaknya yang berada di garis khatulistiwa dengan intensitas penyinaran matahari sepanjang tahun. Selain energi surya, potensi tenaga air, panas bumi, dan bioenergi juga tersebar di berbagai pulau besar seperti Sumatra, Kalimantan, dan Sulawesi.

Meskipun potensi fisiknya sangat besar, realisasi pemanfaatan energi terbarukan di lapangan masih berjalan lambat dan baru menyentuh kisaran 13 hingga 14 persen. Salah satu hambatan teknis yang paling krusial adalah sifat intermitensi pada pembangkit listrik tenaga surya (PLTS) dan tenaga bayu (PLTB). Intermitensi berarti produksi listrik sangat bergantung pada kondisi cuaca dan waktu, sehingga pasokan daya dapat berfluktuasi secara drastis ketika langit mendung atau angin melemah. Kondisi ini menuntut kesiapan jaringan transmisi listrik yang cerdas (smart grid) serta sistem penyimpanan energi berbasis baterai berkapasitas besar agar kestabilan frekuensi listrik tetap terjaga dan tidak memicu pemadaman bergilir.

Di sisi lain, persoalan ketimpangan geografis antara lokasi sumber energi dan pusat permintaan listrik turut memperumit proses transisi. Sebagian besar potensi tenaga air dan panas bumi berada di wilayah terpencil atau di luar Pulau Jawa, sedangkan sekitar 70 persen konsumsi listrik nasional justru terpusat di kawasan industri dan permukiman di Pulau Jawa dan Bali. Membangun jaringan transmisi antar-pulau bertegangan tinggi membutuhkan investasi modal yang sangat besar serta pembebasan lahan yang memakan waktu bertahun-tahun. Akibatnya, sejumlah wilayah di Jawa justru mengalami kondisi kelebihan pasokan listrik (oversupply) dari pembangkit batu bara yang sudah terlanjur dikontrak jangka panjang melalui skema take-or-pay.

Dari aspek ekonomi dan regulasi, biaya investasi awal pembangunan pembangkit energi terbarukan masih dinilai tinggi oleh sebagian pelaku usaha swasta dibandingkan pembangkit fosil yang teknologinya sudah mapan. Walaupun harga panel surya global telah turun lebih dari 80 persen dalam sepuluh tahun terakhir, biaya pendanaan di dalam negeri masih terbebani oleh suku bunga pinjaman yang relatif tinggi serta aturan tingkat komponen dalam negeri (TKDN) yang sempat membatasi masuknya komponen impor berharga murah. Sebagian pengamat ekonomi energi berpendapat bahwa pemerintah perlu segera mereformasi skema subsidi energi fosil dan mengalihkannya menjadi insentif fiskal bagi pengembang energi bersih agar harga jual listrik terbarukan lebih kompetitif.

Namun, kelompok pengamat lain mengingatkan bahwa percepatan penghentian operasi PLTU batu bara secara dini tanpa perencanaan sosial-ekonomi yang matang dapat menimbulkan risiko baru. Sektor pertambangan batu bara masih menjadi tulang punggung pendapatan daerah dan menyerap ratusan ribu tenaga kerja di Kalimantan dan Sumatra. Oleh sebab itu, keberhasilan transisi energi tidak hanya ditentukan oleh kesiapan teknologi kelistrikan semata, tetapi juga oleh kebijakan transisi yang berkeadilan (just transition) yang mampu menciptakan lapangan kerja hijau baru serta menjaga keterjangkauan tarif listrik bagi masyarakat luas.
""".strip()

# Cek jumlah kata untuk memastikan berada di rentang 500 - 800 kata
jumlah_kata = len(teks_materi.split())
print(f"Sumber bacaan : {sumber_bacaan}")
print(f"Jumlah kata   : {jumlah_kata} kata")

if 500 <= jumlah_kata <= 800:
    print("Status        : LAYAK (sesuai rentang 500-800 kata)")
else:
    print("Status        : PERHATIAN (di luar rentang 500-800 kata)")

Sumber bacaan : Contoh Materi Pengembangan - Transisi Energi dan Tantangan Jaringan Listrik Nasional
Jumlah kata   : 503 kata
Status        : LAYAK (sesuai rentang 500-800 kata)


### Langkah 3: Memuat & Menguji Model Summarization dengan (cahya/t5-base-indonesian-summarization-cased)

Uji model menggunakan QWEN

Qwen2.5-1.5B-Instruct adalah model bahasa besar (large language model) berukuran kecil dengan sekitar 1,54 miliar parameter yang dikembangkan oleh Alibaba Cloud dalam seri Qwen2.5 dan telah dioptimalkan untuk mengikuti instruksi (instruction-tuned). Model ini dirilis dengan lisensi Apache 2.0.

In [3]:
import time
from transformers import T5Tokenizer, T5ForConditionalGeneration

SUM_MODEL_ID = "cahya/t5-base-indonesian-summarization-cased"

# 1. Mengukur waktu memuat model
print(f"Memuat model summarization: {SUM_MODEL_ID}...")
start_load = time.time()

tokenizer_sum = T5Tokenizer.from_pretrained(SUM_MODEL_ID)
model_sum = T5ForConditionalGeneration.from_pretrained(SUM_MODEL_ID).to(device)

waktu_load_sum = round(time.time() - start_load, 2)
print(f"Model berhasil dimuat dalam {waktu_load_sum} detik di perangkat: {device.upper()}")

# 2. Fungsi buat_ringkasan (menyiapkan mode pendek, sedang, panjang)
def buat_ringkasan(teks, mode="sedang"):
    # Pengaturan panjang token sesuai mode
    konfigurasi_mode = {
        "pendek": {"min_length": 40, "max_length": 80},
        "sedang": {"min_length": 90, "max_length": 150},
        "panjang": {"min_length": 160, "max_length": 230},
    }
    cfg = konfigurasi_mode.get(mode, konfigurasi_mode["sedang"])

    # Format input T5
    input_ids = tokenizer_sum.encode(
        teks,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)

    # Proses generasi ringkasan
    summary_ids = model_sum.generate(
        input_ids,
        min_length=cfg["min_length"],
        max_length=cfg["max_length"],
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

    ringkasan = tokenizer_sum.decode(summary_ids[0], skip_special_tokens=True)
    return ringkasan

# 3. Uji pertama: Mode SEDANG
start_gen = time.time()
hasil_ringkasan_sedang = buat_ringkasan(teks_materi, mode="sedang")
waktu_gen_sum = round(time.time() - start_gen, 2)

print("\n" + "="*60)
print(f"HASIL RINGKASAN (Mode: SEDANG | Waktu Generasi: {waktu_gen_sum} detik)")
print("="*60)
print(hasil_ringkasan_sedang)
print("="*60)
print(f"Jumlah kata ringkasan: {len(hasil_ringkasan_sedang.split())} kata")

Memuat model summarization: cahya/t5-base-indonesian-summarization-cased...


tokenizer_config.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/793k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  892MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Model berhasil dimuat dalam 16.91 detik di perangkat: CPU

HASIL RINGKASAN (Mode: SEDANG | Waktu Generasi: 44.36 detik)
Transisi energi menuju sumber terbarukan menjadi salah satu agenda utama pembangunan nasional Indonesia dalam dua dekade terakhir. Pemerintah menargetkan porsi energi baru terbarukan (EBT) dalam bauran energi primer mencapai 23 persen pada tahun 2025 dan meningkat hingga lebih dari 31 persen pada 2050. Target ini ditetapkan untuk menekan emisi gas rumah kaca sekaligus mengurangi ketergantungan pada pembangkit listrik tenaga uap (PLTU) berbasis batu bara yang selama ini mendominasi pasokan listrik nasional hingga lebih...
Jumlah kata ringkasan: 73 kata


## Langkah 4: Membersihkan Memori GPU & Memuat Model QG (Qwen/Qwen2.5-1.5B-Instruct)

In [4]:
import gc
import json
import re
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Bersihkan memori GPU dari model summarization sebelumnya
if 'model_sum' in globals():
    del model_sum
if 'tokenizer_sum' in globals():
    del tokenizer_sum
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memori GPU dari model summarization telah dibersihkan.\n")

# 2. Memuat model Question Generation (Qwen2.5-1.5B-Instruct)
QG_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Memuat model QG: {QG_MODEL_ID}...")
start_load_qg = time.time()

tokenizer_qg = AutoTokenizer.from_pretrained(QG_MODEL_ID)
model_qg = AutoModelForCausalLM.from_pretrained(
    QG_MODEL_ID,
    torch_dtype="auto",
    device_map="auto"
)

waktu_load_qg = round(time.time() - start_load_qg, 2)
print(f"Model QG berhasil dimuat dalam {waktu_load_qg} detik di perangkat: {device.upper()}")

# 3. Fungsi buat_soal (1 soal per pemanggilan sesuai rancangan kode)
def buat_soal(teks, kategori="Memahami Informasi"):
    deskripsi_kategori = {
        "Menemukan Informasi": "Menguji kemampuan mencari dan mengidentifikasi fakta atau informasi yang tertulis secara eksplisit di dalam bacaan.",
        "Memahami Informasi": "Menguji kemampuan menyimpulkan makna, hubungan sebab-akibat, atau gagasan antargagasan yang tersirat maupun tersurat dalam bacaan.",
        "Mengevaluasi dan Merefleksi Informasi": "Menguji kemampuan menilai argumen, kelebihan/kelemahan, atau dampak dari gagasan yang disampaikan dalam bacaan."
    }
    panduan_kat = deskripsi_kategori.get(kategori, "")

    system_prompt = (
        "Anda adalah pembuat soal literasi membaca Bahasa Indonesia yang sangat teliti. "
        "Anda HANYA boleh mengeluarkan output berupa JSON valid tanpa teks pengantar atau penutup apa pun."
    )

    user_prompt = f"""Berdasarkan teks bacaan berikut, buatlah tepat 1 (satu) soal pilihan ganda dalam Bahasa Indonesia.

Kategori Soal: {kategori}
Deskripsi Kategori: {panduan_kat}

Aturan Wajib:
1. Pertanyaan harus sesuai dengan kategori soal dan HANYA bersumber dari teks bacaan.
2. Sediakan tepat 5 pilihan jawaban (A, B, C, D, E) yang logis dan homogen.
3. Tentukan 1 kunci jawaban yang benar (hanya huruf A, B, C, D, atau E).
4. Berikan pembahasan singkat mengapa kunci tersebut benar.
5. Sertakan kutipan bukti kalimat yang diambil persis (verbatim) dari teks bacaan.
6. Keluarkan HANYA dalam format JSON persis seperti struktur berikut:

{{
  "kategori": "{kategori}",
  "pertanyaan": "...",
  "opsi": {{
    "A": "...",
    "B": "...",
    "C": "...",
    "D": "...",
    "E": "..."
  }},
  "kunci_jawaban": "A",
  "pembahasan": "...",
  "kutipan_bukti": "..."
}}

Teks Bacaan:
\"\"\"
{teks}
\"\"\"
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    text_input = tokenizer_qg.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer_qg([text_input], return_tensors="pt").to(device)

    start_gen = time.time()
    generated_ids = model_qg.generate(
        **model_inputs,
        max_new_tokens=600,
        temperature=0.3,
        do_sample=True
    )
    waktu_gen = round(time.time() - start_gen, 2)

    # Ambil hanya token baru hasil generasi
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_output = tokenizer_qg.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Coba parsing JSON untuk mengecek validitas struktur
    json_valid = False
    parsed_data = None
    try:
        # Bersihkan jika model membungkus JSON dengan ```json ... ```
        clean_json = re.sub(r"^```json\s*|\s*```$", "", raw_output, flags=re.MULTILINE).strip()
        parsed_data = json.loads(clean_json)
        json_valid = True
    except Exception as e:
        parsed_data = {"error_parsing": str(e), "raw_output": raw_output}

    return {
        "waktu_generasi": waktu_gen,
        "json_valid": json_valid,
        "hasil": parsed_data,
        "raw_output": raw_output
    }

Memori GPU dari model summarization telah dibersihkan.

Memuat model QG: Qwen/Qwen2.5-1.5B-Instruct...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model QG berhasil dimuat dalam 55.11 detik di perangkat: CPU


### Langkah 5: Menjalankan Pengujian 3 Kategori Soal QG

In [5]:
daftar_kategori = [
    "Menemukan Informasi",
    "Memahami Informasi",
    "Mengevaluasi dan Merefleksi Informasi"
]

hasil_uji_qg = []

for idx, kat in enumerate(daftar_kategori, start=1):
    print(f"Sedang membuat soal {idx}/3 — Kategori: {kat}...")
    res = buat_soal(teks_materi, kategori=kat)
    hasil_uji_qg.append(res)

    print("\n" + "="*70)
    print(f"SOAL {idx}: {kat.upper()} (Waktu: {res['waktu_generasi']} detik)")
    print("="*70)

    if res["json_valid"]:
        data = res["hasil"]
        opsi = data.get("opsi", {})
        kunci = data.get("kunci_jawaban", "")
        bukti = data.get("kutipan_bukti", "")

        # Pengecekan otomatis kualitas struktural
        jumlah_opsi = len(opsi)
        bukti_ditemukan = bukti.strip().lower() in teks_materi.lower() if bukti else False

        print(f"[Cek JSON]          : VALID")
        print(f"[Cek Jumlah Opsi]   : {jumlah_opsi} opsi ({list(opsi.keys())})")
        print(f"[Cek Kunci Jawaban] : {kunci} {'(Valid)' if kunci in ['A','B','C','D','E'] else '(TIDAK VALID)'}")
        print(f"[Cek Kutipan Bukti] : {'DITEMUKAN persis dalam teks' if bukti_ditemukan else 'PERINGATAN: Tidak persis / tidak ditemukan di teks'}")
        print("-" * 70)
        print(f"Pertanyaan   : {data.get('pertanyaan')}")
        for huruf, teks_opsi in opsi.items():
            print(f"  {huruf}. {teks_opsi}")
        print(f"Kunci        : {kunci}")
        print(f"Pembahasan   : {data.get('pembahasan')}")
        print(f"Kutipan Bukti: \"{bukti}\"")
    else:
        print("[Cek JSON]          : GAGAL PARSING JSON")
        print("Raw Output Model:")
        print(res["raw_output"])
    print("="*70 + "\n")

Sedang membuat soal 1/3 — Kategori: Menemukan Informasi...

SOAL 1: MENEMUKAN INFORMASI (Waktu: 706.41 detik)
[Cek JSON]          : VALID
[Cek Jumlah Opsi]   : 5 opsi (['A', 'B', 'C', 'D', 'E'])
[Cek Kunci Jawaban] : B (Valid)
[Cek Kutipan Bukti] : DITEMUKAN persis dalam teks
----------------------------------------------------------------------
Pertanyaan   : Berapa persen target EBT yang diprediksi akan dicapai pada tahun 2050?
  A. 23%
  B. 31%
  C. 40%
  D. 50%
  E. 60%
Kunci        : B
Pembahasan   : Pendapatannya ditemukan dalam kalimat 'Target ini ditetapkan untuk menekan emisi gas rumah kaca sekaligus mengurangi ketergantungan pada pembangkit listrik tenaga uap (PLTU) berbasis batu bara yang selama ini mendominasi pasokan listrik nasional hingga lebih dari 60 persen.', dimana 60 persen merupakan nilai yang paling akurat.
Kutipan Bukti: "Target ini ditetapkan untuk menekan emisi gas rumah kaca sekaligus mengurangi ketergantungan pada pembangkit listrik tenaga uap (PLTU) berbasis

### Langkah 6: Mencatat Evaluasi & Membuat model_trial.md Otomatis

In [6]:
import os
from datetime import datetime

# =====================================================================
# SESUAIKAN CATATAN EVALUASI MANUAL DI BAWAH INI BERDASARKAN OUTPUTMU
# =====================================================================
catatan_kualitas_sum = (
    "Model berhasil dijalankan. Ringkasan mode sedang terbaca dengan baik "
    "dalam Bahasa Indonesia, gagasan utama terjaga, dan tidak menambah fakta di luar teks."
)

catatan_kualitas_qg = (
    "Tiga kategori soal berhasil dihasilkan dalam format JSON valid dengan 5 opsi (A-E), "
    "kunci jawaban, pembahasan, dan kutipan bukti yang sesuai dengan teks bacaan."
)

keputusan_model = (
    "LANJUTKAN: Kedua model (cahya/t5-base-indonesian-summarization-cased dan "
    "Qwen/Qwen2.5-1.5B-Instruct) berhasil berjalan lancar di GPU Colab dengan hasil layak "
    "untuk dilanjutkan ke tahap perapian fungsi (src/) dan integrasi UI Streamlit."
)
# =====================================================================

# Mengumpulkan metrik waktu dari variabel pengujian sebelumnya
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
total_waktu_qg = round(sum(item["waktu_generasi"] for item in hasil_uji_qg), 2)
jumlah_json_valid = sum(1 for item in hasil_uji_qg if item["json_valid"])

# Menyusun isi laporan Markdown sesuai format docs/model_trial.md
laporan_md = f"""# Catatan Uji Kelayakan Awal Model (`model_trial.md`)

**Tanggal Pengujian:** {datetime.now().strftime("%Y-%m-%d %H:%M")}
**Notebook Uji:** `notebooks/01_uji_awal_model.ipynb`
**Sumber Bacaan Pengembangan:** {sumber_bacaan} ({jumlah_kata} kata)

---

## 1. Ringkasan Konfigurasi & Hasil Pengujian

| Komponen | Model ID | Perangkat | Waktu Muat | Waktu Generasi | Status & Hasil Pemeriksaan |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Summarization** | `{SUM_MODEL_ID}` | {device.upper()} ({gpu_name}) | {waktu_load_sum} dtk | {waktu_gen_sum} dtk | Berhasil (Mode sedang: {len(hasil_ringkasan_sedang.split())} kata) |
| **Question Generation (QG)** | `{QG_MODEL_ID}` | {device.upper()} ({gpu_name}) | {waktu_load_qg} dtk | {total_waktu_qg} dtk (3 soal) | JSON Valid: {jumlah_json_valid}/3 soal |

**Konfigurasi Library:**
- `torch`: `{torch.__version__}`
- `transformers`: `{transformers.__version__}`

---

## 2. Catatan Detail & Temuan Kesalahan

### A. Summarization (`{SUM_MODEL_ID}`)
- **Catatan Kualitas:** {catatan_kualitas_sum}
- **Hasil Ringkasan (Mode Sedang):**
  > {hasil_ringkasan_sedang}

### B. Question Generation (`{QG_MODEL_ID}`)
- **Catatan Kualitas:** {catatan_kualitas_qg}
- **Rincian Per Kategori Soal:**
"""

for idx, item in enumerate(hasil_uji_qg, start=1):
    kat = daftar_kategori[idx - 1]
    w_gen = item["waktu_generasi"]
    if item["json_valid"]:
        d = item["hasil"]
        opsi_lengkap = len(d.get("opsi", {})) == 5
        bukti = d.get("kutipan_bukti", "")
        bukti_cocok = bukti.strip().lower() in teks_materi.lower() if bukti else False
        laporan_md += f"""
#### Soal {idx}: {kat} ({w_gen} detik)
- **Status JSON:** Valid | **Opsi A–E:** {"Lengkap (5 opsi)" if opsi_lengkap else "Tidak lengkap"} | **Kecocokan Bukti:** {"Persis di teks" if bukti_cocok else "Perlu pengecekan manual"}
- **Pertanyaan:** {d.get("pertanyaan")}
- **Opsi:**
  - A. {d.get("opsi", {}).get("A", "-")}
  - B. {d.get("opsi", {}).get("B", "-")}
  - C. {d.get("opsi", {}).get("C", "-")}
  - D. {d.get("opsi", {}).get("D", "-")}
  - E. {d.get("opsi", {}).get("E", "-")}
- **Kunci Jawaban:** {d.get("kunci_jawaban")}
- **Pembahasan:** {d.get("pembahasan")}
- **Kutipan Bukti:** `"{bukti}"`
"""
    else:
        laporan_md += f"""
#### Soal {idx}: {kat} ({w_gen} detik)
- **Status JSON:** Gagal Parsing
- **Raw Output:** `{item["raw_output"]}`
"""

laporan_md += f"""
---

## 3. Keputusan Sementara

> **{keputusan_model}**

*(Catatan: Untuk demonstrasi aplikasi Streamlit di laptop lokal tanpa GPU CUDA, perlu diperhatikan bahwa waktu inferensi di CPU akan lebih lambat dibandingkan GPU T4 Colab).*
"""

# Simpan ke file docs/model_trial.md di Colab
os.makedirs("docs", exist_ok=True)
output_path = "docs/model_trial.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(laporan_md)

print(f"File laporan berhasil dibuat di: {output_path}\n")
print("="*70)
print(laporan_md)

File laporan berhasil dibuat di: docs/model_trial.md

# Catatan Uji Kelayakan Awal Model (`model_trial.md`)

**Tanggal Pengujian:** 2026-11-19 16:24  
**Notebook Uji:** `notebooks/01_uji_awal_model.ipynb`  
**Sumber Bacaan Pengembangan:** Contoh Materi Pengembangan - Transisi Energi dan Tantangan Jaringan Listrik Nasional (503 kata)

---

## 1. Ringkasan Konfigurasi & Hasil Pengujian

| Komponen | Model ID | Perangkat | Waktu Muat | Waktu Generasi | Status & Hasil Pemeriksaan |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Summarization** | `cahya/t5-base-indonesian-summarization-cased` | CPU (CPU) | 16.91 dtk | 44.36 dtk | Berhasil (Mode sedang: 73 kata) |
| **Question Generation (QG)** | `Qwen/Qwen2.5-1.5B-Instruct` | CPU (CPU) | 55.11 dtk | 2061.04 dtk (3 soal) | JSON Valid: 3/3 soal |

**Konfigurasi Library:**
- `torch`: `2.14.1+cpu`
- `transformers`: `5.22.1`

---

## 2. Catatan Detail & Temuan Kesalahan

### A. Summarization (`cahya/t5-base-indonesian-summarization-cased`)
- **